# BigData 분석

## 1. AIS 데이터셋 기반 Density Heatmap

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import matplotlib.colors as mcolors

# CSV file path
file_path = 'C:\AIS_DATA\AIS dataset\Dynamic_20200212.csv'

# Load the CSV file
try:
    df = pd.read_csv(file_path, encoding='cp949', skiprows=2)
except FileNotFoundError:
    print(f"Error: The file {file_path} was not found. Please check the file path.")
    exit()

# Check for required columns ('위도', '경도')
required_columns = ['위도', '경도']
if not all(col in df.columns for col in required_columns):
    print(f"Error: The CSV file must contain the columns {required_columns}.")
    exit()

# Limit the data range to the waters around the Korean Peninsula
# Set the latitude (Latitude) range to 33° ~ 43°
# Set the longitude (Longitude) range to 124° ~ 132°
lat_min, lat_max = 33, 43
lon_min, lon_max = 124, 132

# Filter the data within the specified range
df_filtered = df[(df['위도'] >= lat_min) & (df['위도'] <= lat_max) &
                 (df['경도'] >= lon_min) & (df['경도'] <= lon_max)].copy()

# Handle cases where no data is filtered
if df_filtered.empty:
    print("Error: No data found within the specified range. Please adjust the range.")
    exit()

# Calculate the 2D histogram (density) based on latitude and longitude data
# Use bins=100 to automatically create appropriate bins
heatmap_data, xedges, yedges = np.histogram2d(df_filtered['경도'], df_filtered['위도'], bins=100)

# Visualize the heatmap using Matplotlib
plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(12, 10))

# Apply logarithmic normalization to the color map
# This enhances the visibility of small values
# We add a small value (1e-5) to avoid log(0)
norm = mcolors.LogNorm(vmin=max(heatmap_data.min(), 1), vmax=heatmap_data.max())

# Use the imshow() function to visualize the 2D histogram
# Set origin='lower' so the y-axis (Latitude) increases from bottom to top
plt.imshow(heatmap_data.T, origin='lower', cmap='YlGnBu', alpha=0.8, extent=[xedges[0], xedges[-1], yedges[0], yedges[-1]], norm=norm)
plt.colorbar(label='Vessel Traffic (Number of Data Points)')

# Mark major ports on the graph
# Approximate latitude/longitude coordinates for each port
ports = {
    'Donghae': (37.5, 129.1),
    'Pohang': (36.0, 129.4),
    'Incheon': (37.4, 126.6),
    'Jeju': (33.5, 126.5),
    'Ulsan': (35.5, 129.4),
    'Yeosu': (34.7, 127.7)
}

# Add markers and text labels for each port
for port_name, (lat, lon) in ports.items():
    plt.plot(lon, lat, 's', color='blue', markersize=10, markeredgecolor='white', markeredgewidth=1)
    plt.text(lon + 0.1, lat, port_name, color='black', fontsize=12, fontweight='bold', ha='left', va='center')

# Set axis labels and title
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title('Density Heatmap of AIS Datasets with Major Ports (Log-Normalized)')

# Save the graph as an image file
plt.tight_layout()
plt.savefig('ais_heatmap_improved_ports.png', dpi=300)
plt.show()

print("The heatmap has been saved as 'ais_heatmap_improved_ports.png' file.")


## 2. SOG 및 COG Distribution 그래프

In [ ]:
# 그래프 설정
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'NanumGothic'  # 한글 폰트 설정 (필요에 따라 다른 폰트명으로 변경)
plt.rcParams['axes.unicode_minus'] = False     # 마이너스 기호 깨짐 방지

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# --- SOG 분포 히스토그램/KDE 플롯 ---
sog_data = df['SOG']
sog_mean = sog_data.mean()
sog_mean = round(sog_mean, 2)
sog_std = sog_data.std()

sns.histplot(sog_data, bins=30, kde=True, ax=axes[0], color='skyblue', edgecolor='black', alpha=0.7)

# 평균과 분산 강조
axes[0].axvline(sog_mean, color='red', linestyle='--', linewidth=2, label=f'Mean: {sog_mean} knots')
axes[0].axvline(sog_mean + sog_std, color='green', linestyle=':', linewidth=2, label=f'Std: {sog_std:.2f} knots')
axes[0].axvline(sog_mean - sog_std, color='green', linestyle=':', linewidth=2)

axes[0].set_title('SOG Distribution', fontsize=16, fontweight='bold')
axes[0].set_xlabel('SOG (knots)', fontsize=12)
axes[0].set_ylabel('Frequency', fontsize=12)
axes[0].legend()
axes[0].grid(True)


# --- COG 분포 극좌표 히스토그램 (Rose Diagram) ---
cog_radians = np.radians(df['COG']) # COG를 라디안으로 변환
ax_polar = fig.add_subplot(1, 2, 2, projection='polar')

# 히스토그램 생성
ax_polar.hist(cog_radians, bins=30, color='royalblue', edgecolor='white', alpha=0.8, density=True)

# 극좌표 플롯 설정
ax_polar.set_theta_zero_location('N') # 북쪽(0도)을 위로 설정
ax_polar.set_theta_direction(-1)      # 시계방향으로 방향 설정 (방위각 일반 표기법)

# 각도 레이블 설정 (0, 45, 90, ..., 315)
ax_polar.set_thetagrids(np.arange(0, 360, 45), labels=['0°', '45°', '90°', '135°', '180°', '225°', '270°', '315°'])

# 제목 및 캡션 설정
ax_polar.set_title('COG Distribution', fontsize=16, fontweight='bold', pad=20)
ax_polar.grid(True)

plt.suptitle('Analysis of AIS datasets ', fontsize=20, fontweight='bold')
plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # 그래프 간 간격 조정
plt.show()

## 3. 시간 간격 그래프

In [ ]:
# --- 그래프 (시간축 시각화) ---
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.family'] = 'NanumGothic'  # 한글 폰트 설정 (필요에 따라 다른 폰트명으로 변경)
plt.rcParams['axes.unicode_minus'] = False     # 마이너스 기호 깨짐 방지

# 시각화할 특정 선박 선택 (샘플 데이터에서는 MMSI 123456789)
target_mmsi = 123456789
ship_df = df[df['MMSI'] == target_mmsi].copy()

# 시간 간격 계산 (초 단위)
ship_df['time_diff'] = ship_df['time'].diff().dt.total_seconds().fillna(0)

# 시각화를 위한 플롯 설정
fig, ax = plt.subplots(figsize=(12, 6))

# 정상적인 데이터 구간을 표시합니다.
# 시간 간격이 1분(60초) 이내인 구간을 정상으로 간주
valid_segments = ship_df[ship_df['time_diff'] <= 60]
ax.plot(valid_segments['time'], valid_segments['longitude'], 'o-', color='blue', label='정상 데이터 구간')

# 결측 및 보간 구간을 시각적으로 표시합니다.
# 시간 간격이 1분을 초과하는 구간을 결측/보간 구간으로 간주
missing_segments = ship_df[ship_df['time_diff'] > 60]
for i in missing_segments.index:
    start_point = ship_df.loc[i-1]
    end_point = ship_df.loc[i]
    
    # 결측 구간을 점선으로 연결하여 보간의 필요성을 시각화합니다.
    ax.plot([start_point['time'], end_point['time']],
            [start_point['longitude'], end_point['longitude']],
            '--', color='red', alpha=0.7)

# 개별 결측치(NaN)를 별도로 표시
nan_points = ship_df[ship_df['latitude'].isnull()]
ax.scatter(nan_points['time'], nan_points['longitude'], marker='x', color='red', s=100, label='결측 데이터(NaN)')

# 그래프 제목 및 레이블 설정
ax.set_title(f'선박 MMSI {target_mmsi} 궤적의 결측 구간', fontsize=16, fontweight='bold')
ax.set_xlabel('시간', fontsize=12)
ax.set_ylabel('경도', fontsize=12)
ax.legend()
ax.grid(True)
plt.show()